In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # 加载.env文件里的变量
print(os.getenv("DEEPSEEK_API_KEY"))  # 现在可以正常读取了
print(os.getenv("LANGSMITH_TRACING"))
print(os.getenv("LANGSMITH_ENDPOINT"))
print(os.getenv("LANGSMITH_API_KEY"))
print(os.getenv("LANGSMITH_PROJECT"))

In [ ]:
import getpass
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

import operator
from typing import Annotated,TypedDict,List
from langgraph.graph import StateGraph, END
from IPython.display import Image, display
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )

class State(TypedDict):
    messages: Annotated[List[str], operator.add]

def chatbot(state:State):
    print(state)
    return {"messages":[llm.invoke(state["messages"])]}

In [ ]:
from langgraph.graph import StateGraph,MessagesState

builder=StateGraph(MessagesState)

builder.add_node("chatbot", lambda state: {"messages": [("assistant", "你好，最帅气的人")]}
)

builder.set_entry_point("chatbot")
builder.set_finish_point("chatbot")

graph=builder.compile()

In [ ]:
graph.invoke({"messages": [("user", "你好，请你介绍一下你自己")]})

In [ ]:
graph.invoke({"messages":["user","hi 3213."]})

{'messages': [HumanMessage(content='你好，请你介绍一下你自己', additional_kwargs={}, response_metadata={}, id='2e1fbb40-36f0-4233-a05f-85a11a6d583c'),
  AIMessage(content='你好，最帅气的人', additional_kwargs={}, response_metadata={}, id='f5c579cb-54a8-4863-8f25-49a571dda349', tool_calls=[], invalid_tool_calls=[])]}

In [ ]:
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage,HumanMessage

msgs1=[HumanMessage(content="你好",id=1)]
msgs2=[AIMessage(content="你好，很高兴认识你",id=2)]

add_messages(msgs1,msgs2)

In [ ]:
msgs1=[HumanMessage(content="你好。",id=1)]
msgs2=[AIMessage(content="你好呀",id=2)]
add_messages(msgs1,msgs2)

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph,START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list,add_messages]

graph_builder=StateGraph(State)

In [ ]:
import getpass
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

import operator
from typing import Annotated,TypedDict,List
from langgraph.graph import StateGraph, END
from IPython.display import Image, display
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )

def chatbot(state:State):
    # print(state)
    return {"messages":[llm.invoke(state["messages"])]}

In [ ]:
graph_builder.add_node("chatbot",chatbot)

graph_builder.add_edge(START,"chatbot")
graph_builder.add_edge("chatbot",END)

graph=graph_builder.compile()

In [ ]:
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
def stream_graph_update(user_input: str):
    for event in graph.stream({"messages":["user",user_input]}):
        print(event)
        print(event.values())
        for value in event.values():
            print("模型回复：",value["messages"][-1].content)

while True:
    try:
        user_input=input("用户提问：")
        if user_input.lower() in ["退出"]:
            print("下次再见！")
            break

        stream_graph_update(user_input)

    except:
        break